# 6.2 数据格式：FP8 家族与 HiF8

## 1. 浮点数的本质：二进制科学计数法

一切浮点格式（FP32/FP16/BF16/FP8/HiF8）都是同一个思想的变体：

```
浮点数 = 符号 × 尾数 × 2^指数
```

十进制里 `123.45 = 1.2345 × 10²`，二进制里 `5.75 = 101.11₂ = 1.0111₂ × 2²`。IEEE 754 把它拆成三段位域：

| 域 | 含义 | 规则 |
|--|--|--|
| S（符号） | 正负 | 0 正 1 负 |
| E（指数） | 2 的幂 | **存储偏移后的值**：真实指数 = E − bias，bias = 2^(k−1) − 1 |
| M（尾数） | 有效数字 | 规格化数隐含前导 1：实际尾数 = 1.M |

规格化数的完整公式：`Value = (−1)^S × 1.M × 2^(E−bias)`。

以 FP32 表示 5.75：`5.75 = 1.0111₂ × 2²` → S=0，E = 2 + 127 = 129 = `10000001₂`，M = `0111000...0`（取小数部分 .0111 补零）。

**为什么这样设计**：指数决定动态范围（能表示多大/多小的数），尾数决定精度；加法/乘法通过「指数对齐 + 尾数运算」即可硬件化。

## 2. 常用格式对比

| 类型 | 位宽 | 符号 | 指数 | 尾数 | 动态范围 | 精度 | 典型用途 |
|--|--|--|--|--|--|--|--|
| FP32 | 32 | 1 | 8 | 23 | ~10³⁸ | ~7 位十进制 | 基准训练、黄金对拍 |
| FP16 | 16 | 1 | 5 | 10 | ~10⁵（max 65504） | ~3 位 | 混合精度训练、推理 |
| BF16 | 16 | 1 | 8 | 7 | 同 FP32 | ~2 位 | LLM 训练（TPU/NPU） |
| INT8 | 8 | 1 | — | — | [−128,127] | 极低 | 推理量化 |
| FP8 E4M3 | 8 | 1 | 4 | 3 | max ≈ 448 | ~1.5 位 | 激活、KV Cache（精度优先） |
| FP8 E5M2 | 8 | 1 | 5 | 2 | max ≈ 57344 | ~1 位 | 权重、梯度（范围优先） |

一句话总结：**FP16 重精度，BF16 重范围，INT8 重效率，FP8 是未来**。其中 E4M3 与 E5M2 的分工是「同样 8 位，指数多一位则范围大、尾数少一位则精度低」的直接体现。

用阶码个数看动态范围更直观：FP16 有 40 个阶码、BF16 有 261 个，E4M3 只有 **18 个**（[−9, 8]）、E5M2 有 32 个（[−16, 15]）——这正是 LLM 训练中 E4M3 必须配 extreme 细粒度 scaling 的原因（见 6.6 节 HiF8 的动机）。

## 3. E8M0：专为 scale 而生的格式

E8M0 是「半个浮点」：8 位全是指数、没有符号和尾数，值恒为 `2^(E−127)`。

它不能表示普通数据，却是完美的**块缩放因子**：

- 只能表示 2 的幂 → 反量化时「乘 scale」退化为**指数加 / 移位**，几乎免费；
- 8 位覆盖 2^(−127) ~ 2^(+127) 的缩放范围，足够任何合理的 amax 归一化。

> 这就是 MX 系格式（MXFP8/MXFP6/MXFP4）的设计：**数据用 E4M3 等低精度格式，每 32 个元素配一个 E8M0 scale**，二者合称块浮点（block floating point）。

## 4. HiF8：动态点位域与锥形精度

HiF8 是昇腾提出的 8-bit **单数据格式**（已捐献给全球计算联盟 GCC，并在 IEEE 启动标准立项）。它与传统浮点的关键差异：

- **动态点位域（Dot）**：用变长前缀码显式指示「阶码存几位、是否 Denormal」，使**指数位和尾数位长度可变**；
- **锥形精度**：数值绝对值靠近 1 时精度高，远离 1 时精度渐变降低（是渐变不是跳变），贴合 AI 数据的高斯分布特征；
- **综合阶码范围 [−22, 15] 共 38 个**，接近 FP16 的 40 个——动态范围几乎是 FP16 级别，而精度持平 FP8；
- 阶码采用原码编码并隐藏 1-bit 固定值，不同位宽的阶码范围不重复，实现无冗余编码；支持零（不区分正负零）、NaN、±Inf 四类特殊值。

**与 MXFP8 的本质区别**：MXFP8 是「数据 + 32 元素一组的块 scale」，HiF8 是「数据自带动态范围，最简只需 per-tensor scale」。训练时 HiF8 因此无需按行/按列分别量化激活、权重和梯度，且 delayed-scaling 下 loss 曲线更稳。

## 5. 黄金法则：低精度存储，高精度计算

FP8 的尾数只有 2~3 位，连 1.25 和 1.375 都难以区分。两个 FP8 相乘：

- 448 × 448 = 200704，**远超** E4M3 的 max 448 → 强制存回 FP8 变 inf；
- 0.0156 × 0.0156 ≈ 0.00024，**低于**最小正规数 → 下溢为 0。

所以矩阵乘的**累加器必须是 FP32**（`C = A @ B + C` 中 A、B 为 FP8，累加在 FP32 上），这与 int8 × int8 用 int32 累加防止溢出（255×255=65025 > 255）完全同构。现代 AI 硬件的黄金法则：

> **低精度用于「存储」和「传输」，高精度用于「计算」。**

即使最终把结果转回 FP8 输出，中间累加也走 FP32，否则模型会崩。

![8-bit 格式位布局](images/fp8_formats.svg)

## 小测验

1. FP8 E4M3 的 bias 是多少？它能表示的最大规格化数是多少（数量级）？
2. E4M3 与 E5M2 各有多少个阶码？为什么 LLM 训练里普遍只用 E4M3？这带来什么代价？
3. E8M0 为什么只能表示 2 的幂？这一性质让 MX 系格式的反量化退化成什么运算？
4. HiF8 的「锥形精度」是什么含义？它与 MXFP8 的块浮点在设计哲学上差在哪？
5. 为什么 FP8 × FP8 的累加器必须是 FP32？用 E4M3 的具体数字举一个溢出的例子。

In [ ]:
# 动手：用 numpy 「手工」解码 E4M3，观察它的表示能力
import numpy as np

def e4m3_decode(bits):
    # 解码 uint8 为 E4M3（e4m3fn 变体：E=15 无 Inf，仅 M=7 为 NaN；bias=7）
    S = (bits >> 7) & 0x1
    E = (bits >> 3) & 0xF
    M = bits & 0x7
    if E == 15 and M == 7:
        return np.nan                       # 唯一的 NaN 编码
    if E == 0:
        val = (M / 8.0) * (2.0 ** -6)       # subnormal：0.M × 2^-6
    else:
        val = (1.0 + M / 8.0) * (2.0 ** (E - 7))   # normal：1.M × 2^(E-bias)
    return -val if S else val

codes = np.arange(256, dtype=np.uint8)
vals = np.array([e4m3_decode(c) for c in codes])
normal = vals[np.isfinite(vals)]
print(f'E4M3 有限值个数（正负合计）: {len(normal)}')
print(f'最小正 subnormal: {normal[normal > 0].min():.6f}（理论 2^-9 = 0.001953）')
print(f'最小正规格化数  : {normal[normal >= 2**-6].min():.6f}（理论 2^-6 = 0.015625）')
print(f'最大正规格化数  : {normal.max():.1f}（e4m3fn 无 Inf，E=15 时 max=448）')

# 锥形观察：不同量级下的「最近可表示数」间隔
for x in [0.02, 0.1, 1.0, 5.0, 100.0, 400.0]:
    grid = normal[(normal > 0) & (normal < x * 2)]
    near = grid[np.argmin(np.abs(grid - x))]
    idx = np.where(grid == near)[0][0]
    step = grid[idx + 1] - grid[idx] if idx + 1 < len(grid) else np.nan
    print(f'x = {x:>6}: 最近可表示 {near:>9.4f}, 该量级的间隔 ≈ {step:>9.6f}')

In [ ]:
# 动手：模拟 E4M3 量化-反量化的舍入误差（量化误差是 6.3 节的伏笔）
def e4m3_quantize(x):
    # 编码-再解码：把 float32 舍到最近的 E4M3 可表示值（简化实现）
    x = np.asarray(x, dtype=np.float32)
    out = np.zeros_like(x)
    for i, v in enumerate(x.ravel()):
        grid = normal[np.isfinite(normal)]
        j = np.argmin(np.abs(grid - v))          # 最近邻舍入
        out.ravel()[i] = grid[j] if np.isfinite(grid[j]) else v
    return out

x = np.array([0.156, 1.25, 1.37, 3.99, 448.0, -0.03], dtype=np.float32)
xq = e4m3_quantize(x)
for orig, q in zip(x, xq):
    err = abs(q - orig) / (abs(orig) + 1e-12)
    print(f'{orig:>9.3f} -> {q:>9.3f}   相对误差 {err:.3%}')
print()
print('观察：1.25 / 1.37 一类的近邻数几乎无法区分（尾数只有 3 位）；')
print('     大数附近的绝对间隔大，但相对间隔（ulp/value）不变——这是浮点量化的特点。')